In [ ]:
import json
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions

In [ ]:
# Load the raw JSON dataset containing diseases
data_path = Path("../data/raw/diseases.json")

In [9]:
# Load data
with open(data_path, "r", encoding="utf-8") as f:
    diseases = json.load(f)

In [ ]:
# Transform each disease dictionary into a structured text for the RAG
documents = []
metadatas = []

for disease in diseases:
    # Create a rich textual description combining symptoms, lab results, and treatments
    doc_text = f"""
    Disease Name: {disease['name']}
    Visible Symptoms: {', '.join(disease['visible_symptoms'])}
    Hidden/Felt Symptoms: {', '.join(disease['hidden_symptoms'])}
    Laboratory Results: {disease['lab_results']}
    Recommended Treatment: {disease['treatment']}
    """
    documents.append(doc_text.strip())
    
    # Store metadata to easily track the original ID and name
    metadatas.append({"id": disease["id"], "name": disease["name"]})

In [23]:
print(f" Loaded and structured {len(documents)} disease profiles successfully!")
print("\n--- Example of generated document ---")
print(documents[0])

 Loaded and structured 3 disease profiles successfully!

--- Example of generated document ---
Disease Name: Lupus érythémateux systémique
    Visible Symptoms: Éruption cutanée en ailes de papillon sur le nez et les joues, Fatigue chronique intense
    Hidden/Felt Symptoms: Douleurs articulaires migratoires, Sensibilité extrême au soleil (photosensibilité)
    Laboratory Results: Anticorps anti-ANA positifs, VS (Vitesse de Sédimentation) élevée
    Recommended Treatment: Corticostéroïdes et hydroxychloroquine


In [ ]:
# We initialize ChromaDB client with local persistence (saves vectors in chroma_db/ folder)
chroma_client = chromadb.PersistentClient(path="../chroma_db")

# We use a multilingual embedding model to properly handle French medical descriptions
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# Create or get a Chroma collection for our medical cases
collection_name = "mystery_diseases"
collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_fn
    metadata={"hnsw:space": "cosine"}  # We use cosine similarity instead of L2 as we want to focus on text similarities
)



In [ ]:
# Extract current IDs from our JSON metadatas
current_json_ids = [meta["id"] for meta in metadatas]
current_json_set = set(current_json_ids)

# Clean up deleted items (remove IDs present in DB but no longer in the JSON)
existing_db_set = set(collection.get()["ids"])

# Find elements in the DB that are no longer in the JSON using set difference
ids_to_delete = list(existing_db_set - current_json_set)

if ids_to_delete:
    collection.delete(ids=ids_to_delete)
    print(f" Cleaned up {len(ids_to_delete)} deleted disease(s) from ChromaDB.")

# UPSERT: Add new items or update existing ones automatically
collection.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=current_json_ids
)

print(f" Successfully synchronized! Total active disease profiles in ChromaDB: {collection.count()}")

 Successfully synchronized! Total active disease profiles in ChromaDB: 3


In [27]:
# Test a semantic query simulating a player's clue
query = "Le patient a des urines foncées et des douleurs au ventre"

results = collection.query(
    query_texts=[query],
    n_results=1  # Get the top 1 match
)

print(f"Query: '{query}'")
print(f"Matched Disease: {results['metadatas'][0][0]['name']}")
print(f"Retrieved Context Snippet:\n{results['documents'][0][0][:200]}...")

Query: 'Le patient a des urines foncées et des douleurs au ventre'
Matched Disease: Porphyrie aiguë intermittente
Retrieved Context Snippet:
Disease Name: Porphyrie aiguë intermittente
    Visible Symptoms: Urines de couleur rouge sombre / porto, État confusionnel
    Hidden/Felt Symptoms: Douleurs abdominales aiguës inexpliquées, Faibless...
